In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR2TOT_F.xpt"
csv_file = DATA_DIR / "DR2TOT_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve exactly what pandas decoded from XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 DR2TOT_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DR2DRSTZ",
    "DR2EXMER",
    "DRABF",
    "DRDINT",
    "DR2DBIH",
    "DR2DAY",
    "DR2LANG",
    "DR2MNRSP",
    "DR2HELPD",
    "DR2TNUMF",
    "DR2TKCAL",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TCHOL",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TRET",
    "DR2TVARA",
    "DR2TACAR",
    "DR2TBCAR",
    "DR2TCRYP",
    "DR2TLYCO",
    "DR2TLZ",
    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TFOLA",
    "DR2TFA",
    "DR2TFF",
    "DR2TFDFE",
    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",
    "DR2TCALC",
    "DR2TPHOS",
    "DR2TMAGN",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSODI",
    "DR2TPOTA",
    "DR2TSELE",
    "DR2TCAFF",
    "DR2TTHEO",
    "DR2TALCO",
    "DR2TMOIS",
    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",
    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",
    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",
    "DR2_300",
    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ",
    "DR2TWS"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DR2TOT_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10253:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 83:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,253 rows x 83 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Do NOT globally replace tiny values until the exact
# variable/count pattern is validated.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# This exact pattern was found in the attached
# 2009-2010 DR2TOT_F.xpt.
#
# CDC ranges for these nutrient variables allow zero.
#
# The program STOPS if another copy of DR2TOT_F
# has a different pattern.
#

EXPECTED_TINY_COUNTS = {

    "DR2DBIH": 8,

    "DR2TNUMF": 1,

    "DR2TKCAL": 5,

    "DR2TPROT": 5,
    "DR2TCARB": 6,
    "DR2TSUGR": 7,
    "DR2TFIBE": 95,
    "DR2TTFAT": 6,
    "DR2TSFAT": 7,
    "DR2TMFAT": 7,
    "DR2TPFAT": 7,

    "DR2TCHOL": 58,

    "DR2TATOC": 7,
    "DR2TATOA": 6836,

    "DR2TRET": 70,
    "DR2TVARA": 19,
    "DR2TACAR": 1051,
    "DR2TBCAR": 127,
    "DR2TCRYP": 1021,
    "DR2TLYCO": 2230,
    "DR2TLZ": 118,

    "DR2TVB1": 6,
    "DR2TVB2": 5,
    "DR2TNIAC": 6,
    "DR2TVB6": 6,

    "DR2TFOLA": 6,
    "DR2TFA": 185,
    "DR2TFF": 88,
    "DR2TFDFE": 6,

    "DR2TCHL": 5,
    "DR2TVB12": 16,
    "DR2TB12A": 5187,
    "DR2TVC": 35,
    "DR2TVD": 98,
    "DR2TVK": 14,

    "DR2TCALC": 1,
    "DR2TPHOS": 5,
    "DR2TMAGN": 1,
    "DR2TIRON": 6,
    "DR2TZINC": 2,
    "DR2TCOPP": 1,
    "DR2TSODI": 1,
    "DR2TPOTA": 5,
    "DR2TSELE": 5,

    "DR2TCAFF": 2280,
    "DR2TTHEO": 4349,
    "DR2TALCO": 7301,
    "DR2TMOIS": 1,

    "DR2TS040": 376,
    "DR2TS060": 458,
    "DR2TS080": 187,
    "DR2TS100": 74,
    "DR2TS120": 25,
    "DR2TS140": 9,
    "DR2TS160": 7,
    "DR2TS180": 8,

    "DR2TM161": 32,
    "DR2TM181": 7,
    "DR2TM201": 97,
    "DR2TM221": 2935,

    "DR2TP182": 7,
    "DR2TP183": 7,
    "DR2TP184": 3784,
    "DR2TP204": 225,
    "DR2TP205": 1071,
    "DR2TP225": 905,
    "DR2TP226": 1209,

    "DR2_320Z": 2112,
    "DR2_330Z": 4708,
    "DR2BWATZ": 5335
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DR2TOT_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 54890:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated DR2TOT_F pattern."
)


# ============================================================
# 6. RESTORE VALIDATED ZERO VALUES
# ============================================================

#
# Restore ONLY exact TINY_VALUE occurrences in
# the validated variables above.
#
# This does NOT round or modify legitimate
# non-zero dietary decimals.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 54890:

    raise ValueError(
        "Incorrect total number of zero restorations."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the final
# corrected dataset SHOULD contain.
#
# It differs from the original source only where
# validated tiny-zero values were restored.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. FIND GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL DIETARY FIDELITY RULE
# ------------------------------------------------------------
#
# Dietary decimals MUST remain unchanged.
#
# NO rounding.
# NO truncation.
# NO blanket Int64 conversion.
#

print("\n--- GENUINE DECIMAL CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )


# ============================================================
# 9. EXPECTED GENUINE DECIMAL VARIABLES
# ============================================================

#
# These 50 variables contain legitimate decimal
# values in the attached DR2TOT_F file.
#
# They MUST remain floating-point values.
#

DECIMAL_COLUMNS = [

    "WTDRD1",
    "WTDR2D",

    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",

    "DR2TATOC",
    "DR2TATOA",

    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",

    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",

    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSELE",

    "DR2TALCO",
    "DR2TMOIS",

    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",

    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",

    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",

    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ"
]


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Genuine decimal-variable pattern differs "
        "from validated DR2TOT_F.\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{fractional_columns}"
    )


print(
    "PASS: Genuine decimal variables:",
    len(DECIMAL_COLUMNS)
)


print(
    "\nThese variables will NOT be rounded "
    "or converted to Int64."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ONLY PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# This is the key Dietary rule.
#
# Decimal columns remain float.
#
# Only columns that contain no genuine fractions
# are converted to Int64.
#
# NO .round() is used.
#

INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number fields formatted "
    "without unnecessary .0."
)


print(
    "PASS: Dietary decimal fields remain unchanged."
)


# ============================================================
# 12. CDC CHECK - DR2DRSTZ
# ============================================================

#
# CDC:
#
# 1 = Reliable                       8,288
# 2 = Not reliable                      62
# 4 = Reported breast-milk             118
# 5 = Not done                       1,785
#

EXPECTED_DR2DRSTZ = {
    1: 8288,
    2: 62,
    4: 118,
    5: 1785
}


actual = (
    df["DR2DRSTZ"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- DR2DRSTZ ---")


if actual != EXPECTED_DR2DRSTZ:

    raise ValueError(
        "DR2DRSTZ differs from CDC.\n"
        f"Expected: {EXPECTED_DR2DRSTZ}\n"
        f"Actual: {actual}"
    )


print(
    "PASS: DR2DRSTZ matches CDC."
)


# ============================================================
# 13. CDC CHECK - WTDR2D
# ============================================================

#
# CDC:
#
# Valid two-day sample weight records = 8,406
# Missing                             = 1,847
#
# This is a genuine DECIMAL variable.
#

s = df["WTDR2D"]


print("\n--- WTDR2D ---")


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.notna()
        .sum()
    )
    == 8406
    and
    int(
        s.isna()
        .sum()
    )
    == 1847
):

    raise ValueError(
        "WTDR2D differs from CDC."
    )


print(
    "PASS: WTDR2D count matches CDC."
)


print(
    "PASS: WTDR2D decimal values remain unchanged."
)


# ============================================================
# 14. CDC CHECK - DR2DBIH
# ============================================================

#
# CDC:
#
# Range   = -28 to 165
# Count   = 8,429
# Missing = 1,824
#
# The attached XPT contains 8 tiny values
# representing zero.
#

s = df["DR2DBIH"]


print("\n--- DR2DBIH ---")


print(
    "-28 to 165:",
    int(
        s.between(
            -28,
            165
        ).sum()
    )
)


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            -28,
            165
        ).sum()
    )
    == 8429
    and
    int(
        s.isna()
        .sum()
    )
    == 1824
):

    raise ValueError(
        "DR2DBIH differs from CDC."
    )


print(
    "PASS: DR2DBIH matches CDC."
)


# ============================================================
# 15. CDC CHECK - DR2TNUMF
# ============================================================

#
# CDC:
#
# Number of foods:
#
# 0 to 48 = 8,406
# Missing = 1,847
#
# Zero is legitimate.
#

s = df["DR2TNUMF"]


print("\n--- DR2TNUMF ---")


print(
    "0-48:",
    int(
        s.between(
            0,
            48
        ).sum()
    )
)


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            48
        ).sum()
    )
    == 8406
    and
    int(
        s.isna()
        .sum()
    )
    == 1847
):

    raise ValueError(
        "DR2TNUMF differs from CDC."
    )


print(
    "PASS: DR2TNUMF matches CDC."
)


# ============================================================
# 16. CDC CHECK - DR2TKCAL
# ============================================================

#
# CDC:
#
# Energy:
#
# 0 to 8976 kcal = 8,288
# Missing        = 1,965
#
# Zero is legitimate.
#

s = df["DR2TKCAL"]


if not (
    int(
        s.between(
            0,
            8976
        ).sum()
    )
    == 8288
    and
    int(
        s.isna()
        .sum()
    )
    == 1965
):

    raise ValueError(
        "DR2TKCAL differs from CDC."
    )


print(
    "PASS: DR2TKCAL matches CDC."
)


# ============================================================
# 17. CDC CHECK - DR2TPROT
# ============================================================

#
# Protein is a genuine decimal field.
#
# CDC:
#
# 0 to 555.67 gm = 8,288
# Missing        = 1,965
#

s = df["DR2TPROT"]


if not (
    int(
        s.between(
            0,
            555.67
        ).sum()
    )
    == 8288
    and
    int(
        s.isna()
        .sum()
    )
    == 1965
):

    raise ValueError(
        "DR2TPROT differs from CDC."
    )


print(
    "PASS: DR2TPROT matches CDC."
)


print(
    "PASS: DR2TPROT decimals preserved."
)


# ============================================================
# 18. CDC CHECK - DR2TALCO
# ============================================================

#
# IMPORTANT:
#
# DR2TALCO is a genuine decimal variable.
#
# CDC:
#
# 0 to 266.7 gm = 8,288
# Missing       = 1,965
#
# The attached XPT has 7,301 tiny zero values.
#

s = df["DR2TALCO"]


print("\n--- DR2TALCO ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "0-266.7:",
    int(
        s.between(
            0,
            266.7
        ).sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.eq(0)
        .sum()
    )
    == 7301
    and
    int(
        s.between(
            0,
            266.7
        ).sum()
    )
    == 8288
    and
    int(
        s.isna()
        .sum()
    )
    == 1965
):

    raise ValueError(
        "DR2TALCO differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: DR2TALCO matches CDC."
)


print(
    "PASS: Non-zero alcohol decimals remain unchanged."
)


# ============================================================
# 19. CDC CHECK - DR2_300
# ============================================================

#
# CDC:
#
# 1 = Much more than usual      527
# 2 = Usual                   6,916
# 3 = Much less than usual      964
# 7 = Refused                     2
# 9 = Don't know                 59
# Missing                    1,785
#

EXPECTED_DR2_300 = {
    1: 527,
    2: 6916,
    3: 964,
    7: 2,
    9: 59
}


actual = (
    df["DR2_300"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_DR2_300
    or
    int(
        df["DR2_300"]
        .isna()
        .sum()
    )
    != 1785
):

    raise ValueError(
        "DR2_300 differs from CDC."
    )


print(
    "PASS: DR2_300 matches CDC."
)


# ============================================================
# 20. CDC CHECK - WATER VARIABLES
# ============================================================

#
# DR2_320Z:
#
# Total plain water
# CDC range 0 to 8887.5
#
# Genuine decimal variable.
#

s = df["DR2_320Z"]


if not (
    int(
        s.between(
            0,
            8887.5
        ).sum()
    )
    == 8406
    and
    int(
        s.isna()
        .sum()
    )
    == 1847
):

    raise ValueError(
        "DR2_320Z differs from CDC."
    )


print(
    "PASS: DR2_320Z matches CDC."
)


#
# DR2_330Z:
#
# Total tap water
# CDC range 0 to 8591.25
#

s = df["DR2_330Z"]


if not (
    int(
        s.between(
            0,
            8591.25
        ).sum()
    )
    == 8406
    and
    int(
        s.isna()
        .sum()
    )
    == 1847
):

    raise ValueError(
        "DR2_330Z differs from CDC."
    )


print(
    "PASS: DR2_330Z matches CDC."
)


#
# DR2BWATZ:
#
# Total bottled water
# CDC range 0 to 8887.5
#

s = df["DR2BWATZ"]


if not (
    int(
        s.between(
            0,
            8887.5
        ).sum()
    )
    == 8406
    and
    int(
        s.isna()
        .sum()
    )
    == 1847
):

    raise ValueError(
        "DR2BWATZ differs from CDC."
    )


print(
    "PASS: DR2BWATZ matches CDC."
)


print(
    "PASS: All water decimals remain unchanged."
)


# ============================================================
# 21. CDC CHECK - DR2TWS
# ============================================================

EXPECTED_DR2TWS = {
    1: 5340,
    2: 836,
    3: 41,
    4: 1942,
    91: 1,
    99: 307
}


actual = (
    df["DR2TWS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_DR2TWS
    or
    int(
        df["DR2TWS"]
        .isna()
        .sum()
    )
    != 1786
):

    raise ValueError(
        "DR2TWS differs from CDC."
    )


print(
    "PASS: DR2TWS matches CDC."
)


# ============================================================
# 22. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected changes from original XPT:
#
# exact TINY_VALUE -> numeric zero
#
# ONLY in the 70 validated variables.
#
# Nothing else may change numerically.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy(
                dtype=float
            )
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 83 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format.
# NO rounding.
#
# Whole-number variables are Int64.
#
# Genuine Dietary decimal variables remain float.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 24. RE-READ EXPORTED CSV
# ============================================================

#
# round_trip precision is important for verifying
# exact numeric fidelity of decimal columns.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 25. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 83 CSV columns match "
    "expected corrected XPT values."
)


# ============================================================
# 26. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 27. CHECK CSV FOR REMAINING TINY VALUES
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 28. WHOLE-NUMBER .0 FORMAT CHECK
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT search the entire Dietary CSV for ".0".
#
# Genuine decimal variables may legitimately
# contain values such as:
#
#     12.0
#     0.0
#     8591.25
#
# We check ONLY INTEGER_COLUMNS.
#

print("\n--- WHOLE-NUMBER FORMAT CHECK ---")


dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        for col in INTEGER_COLUMNS:

            value = row[col]


            if (
                value != ""
                and
                value.endswith(".0")
            ):

                dot_zero_errors.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


                if len(
                    dot_zero_errors
                ) >= 20:

                    break


        if len(
            dot_zero_errors
        ) >= 20:

            break


print(
    "Unnecessary '.0' errors:",
    len(
        dot_zero_errors
    )
)


if dot_zero_errors:

    print(
        dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains "
        "in whole-number fields."
    )


print(
    "PASS: SEQN and whole-number fields "
    "have no unnecessary .0."
)


# ============================================================
# 29. EXACT DIETARY DECIMAL FIDELITY
# ============================================================

#
# Every genuine decimal column must survive:
#
# XPT
# -> validated zero restoration
# -> CSV
# -> pandas re-read
#
# with NO numeric change.
#

print("\n--- DIETARY DECIMAL FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Dietary decimal values changed in:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All",
    len(DECIMAL_COLUMNS),
    "genuine decimal variables preserved exactly."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

#
# No scientific notation should remain after
# tiny-zero correction.
#

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "DR2TOT_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables preserved:",
    len(DECIMAL_COLUMNS)
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: 0 expected"
)


print(
    "Dietary decimal values changed: 0 expected"
)

NHANES 2009-2010 DR2TOT_F XPT -> CSV
Rows: 10,253
Columns: 83
PASS: Structure = 10,253 rows x 83 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 54,890

Tiny values by variable:
DR2TALCO    7301
DR2TATOA    6836
DR2BWATZ    5335
DR2TB12A    5187
DR2_330Z    4708
DR2TTHEO    4349
DR2TP184    3784
DR2TM221    2935
DR2TCAFF    2280
DR2TLYCO    2230
DR2_320Z    2112
DR2TP226    1209
DR2TP205    1071
DR2TACAR    1051
DR2TCRYP    1021
DR2TP225     905
DR2TS060     458
DR2TS040     376
DR2TP204     225
DR2TS080     187
DR2TFA       185
DR2TBCAR     127
DR2TLZ       118
DR2TVD        98
DR2TM201      97
DR2TFIBE      95
DR2TFF        88
DR2TS100      74
DR2TRET       70
DR2TCHOL      58
DR2TVC        35
DR2TM161      32
DR2TS120      25
DR2TVARA      19
DR2TVB12      16
DR2TVK        14
DR2TS140       9
DR2DBIH        8
DR2TS180       8
DR2TM181       7
DR2TSFAT       7
DR2TATOC       7
DR2TSUGR       7
DR2TPFAT       7
DR2TS160       7
DR2TP183      